# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a comprehensive guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and pretty-print metadata
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

To ensure all references are robust and reproducible, we refer to entities by their Croissant `@id` fields. Let's list the record sets in the dataset schema, and their respective fields and field `@id`s.

In [ ]:
# Utility function to extract record set info from the Croissant metadata
def get_record_sets(md):
    # The record sets are typically in the '@graph' root element in Croissant JSON-LD
    if '@graph' in md:
        graph = md['@graph']
    else:
        graph = md.get('graph', [])

    record_sets = []
    for entry in graph:
        types = entry.get('@type', [])
        if isinstance(types, str):
            types = [types]
        if 'cr:RecordSet' in types or 'RecordSet' in types:
            record_sets.append(entry)
    return record_sets

# We'll parse all @id and fields for each record set
metadata_dict = dataset.metadata.to_json_ld()
record_sets = get_record_sets(metadata_dict)
if not record_sets:
    print("No 'cr:RecordSet' found in @graph. Attempting fallback to top-level 'recordSet' property.")
    record_sets = metadata_dict.get('recordSet', [])

if isinstance(record_sets, dict):
    record_sets = [record_sets]

for idx, rs in enumerate(record_sets):
    rs_id = rs.get('@id', None)
    rs_name = rs.get('schema:name', rs.get('name', f'RecordSet {idx+1}'))
    print(f"Record Set {idx+1}: {rs_name} @id: {rs_id}")
    fields = rs.get('cr:field', rs.get('field', []))
    # fields may be dict or list
    if isinstance(fields, dict): fields = [fields]
    if fields:
        print('  Fields:')
        for f in fields:
            # Sometimes field may be a string (just @id), sometimes an object
            if isinstance(f, str):
                print(f'    - {f}')
            elif isinstance(f, dict):
                print(f'    - {f.get("@id", "[no id]")}: {f.get("schema:name", f.get("name", ""))}')
    else:
        print('  No fields found for this record set.')


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All table access is referenced by record set and field `@id` according to the Croissant schema.

In [ ]:
# For this dataset, the main record set is likely the only tabular data (proteins), so we enumerate all available record sets
# We'll extract all found record sets with their @ids

import pprint
record_set_ids = []
metadata_dict = dataset.metadata.to_json_ld()
record_sets = get_record_sets(metadata_dict)
for rs in record_sets:
    rs_id = rs.get('@id')
    if rs_id is not None:
        record_set_ids.append(rs_id)

# If no record sets found, fail gracefully
if not record_set_ids:
    raise RuntimeError("No RecordSet @id found in metadata.")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        print(f"Loading records for RecordSet {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        print(df.head(3))
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, categorizing, or grouping data. All fields are referenced using their Croissant `@id` values from the schema.

In [ ]:
# Choose a record set and pick a numeric field for demonstration

# We'll choose the first record set for exploration
active_record_set_id = record_set_ids[0]
df = dataframes[active_record_set_id]
print(f"Working with record set: {active_record_set_id}")

# Print all possible field/column names
print("Available columns:", df.columns.tolist())

# Try to pick a reasonable/protein abundance-related numeric field
possible_numeric_fields = [c for c in df.columns if df[c].dtype in ['float64', 'int64', 'float32', 'int32'] or c.lower() in ['abundance', 'mw', 'coverage', 'molecular_weight', 'peptide_count', 'normalized_abundance']]
print("Possible numeric fields:", possible_numeric_fields)

# Let's pick the first available numeric field, or fallback to user specification
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Selected numeric field: {numeric_field}")
else:
    raise RuntimeError("No suitable numeric field found in the dataset.")

# Remove missing or placeholder values if present
df_clean = df.copy()
df_clean = df_clean[pd.to_numeric(df_clean[numeric_field], errors='coerce').notnull()]
df_clean[numeric_field] = pd.to_numeric(df_clean[numeric_field], errors='coerce')

# Example: Only keep proteins with this field greater than a defined threshold
threshold = df_clean[numeric_field].quantile(0.9) if df_clean[numeric_field].max() > 100 else 10
filtered_df = df_clean[df_clean[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field if present (e.g. sample, condition, etc)
possible_categorical_fields = [c for c in df_clean.columns if df_clean[c].dtype == object and c != numeric_field]
group_field = possible_categorical_fields[0] if possible_categorical_fields else None

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot the distribution of the selected numeric field, optionally grouped by a chosen category.

In [ ]:
# Basic visualization
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df_clean[numeric_field], kde=True, bins=30)
plt.title(f"Distribution of {numeric_field} in record set {active_record_set_id}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} distribution by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and examine the FAIR² mass spectrometry dataset using the `mlcroissant` library, referencing all schema elements by their Croissant `@id`. We inspected available record sets, extracted data, performed basic filtering and normalization using field `@id`s, and visualized the results. Further analysis can build upon these templates for detailed biological insight or machine learning workflows.